## BBM 409 - Programming Assignment 1

###  Mustafa Ege 2210356088

## Understanding Perceptron Learning Algorithm (100 points)

### 1. Data Loading and Exploration (5 points)
The goal of this task is to distinguish between two raisin varieties — **Kecimen** and **Besni** — based on geometric features extracted from their images.

The dataset contains **900 samples** and **7 numerical features**, each describing a shape-related property of the raisins:

- **Area:** Total number of pixels inside the raisin boundary  
- **MajorAxisLength:** Length of the longest axis of the raisin shape  
- **MinorAxisLength:** Length of the shortest axis  
- **Eccentricity:** Measures how elongated the raisin is (values closer to 1 indicate a more stretched shape)  
- **ConvexArea:** Area of the smallest convex polygon that can contain the raisin  
- **Extent:** Ratio of the raisin area to the area of its bounding box  
- **Perimeter:** The total boundary length of the raisin shape  

The target variable represents the **class label** of each sample:
- `Kecimen` — 450 samples  
- `Besni` — 450 samples  

The dataset is **balanced** and contains **no missing values**, making it ready for further preprocessing and model training without additional cleaning steps.

From the summary statistics, we can observe that the feature values (such as area, perimeter, and axis lengths) vary across a wide range, suggesting the raisins differ significantly in their size and shape. This indicates that **feature scaling** will likely be important before applying the perceptron, since features are on different scales (e.g., `Area` ranges in the tens of thousands, while `Extent` is between 0 and 1).

In [ ]:
## Import necessary libraries
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split 
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


In [ ]:
# fetch dataset
raisin = fetch_ucirepo(id=850)

# data (as pandas dataframes)
X_df = raisin.data.features
y_df = raisin.data.targets

# metadata
print(raisin.metadata)

# variable information
print(raisin.variables)

In [ ]:
# Preview first few rows
print("\nFirst 5 rows:")
print(X_df.head())

# Dataset info
print("\nDataset info:")
print(X_df.info())

# Summary statistics
print("\nSummary statistics:")
print(X_df.describe())

# Unique class labels
print("\nTarget variable distribution:")
print(y_df['Class'].value_counts())

# Check for missing values
print("\nMissing values per column:")
print(X_df.isnull().sum())


Convert the data frame into a Numpy array.

In [ ]:
X_np = X_df.to_numpy()
y_np = y_df.to_numpy().ravel()

print("\nConverted to NumPy arrays:")
print("X shape:", X_np.shape)
print("y shape:", y_np.shape)

### 2. Data Preprocessing (5 points)

#### Feature Scaling Techniques:
Feature scaling is needed on our data because the features like 'Area' and 'Eccentricity' are on completely on different scales one being around thousands and the other being smaller than 1. Features with large absolute values would dominate the distance/weighted sum calculation, preventing the perceptron from learning an optimal weight vector.

To ensure all features contribute proportionally to the decision boundary, we apply scaling.

We will explore three common techniques and select the one that yields the best training accuracy for the final model in the **next section**. The scaling methods we will try are below:

- **Standardization:** It scales the features to have a mean of 0 and a standard deviation of 1. Centers the data and avoids features with large variances disproportionately influencing the model's convergence.
- **Min-Max Scaling:** It scales the features to a fixed range, typically between 0 and 1. Ensures all features have an identical scale, which can be useful when weights need to be small and constrained.
- **Robust Scaling:** Scales features using the Median and the Interquartile Range (IQR) (the difference between the 75th and 25th percentiles). This method is robust to outliers, as the median and IQR are less affected by extreme values than the mean and standard deviation are.

The labels of raisins are in categorical format as Besni and Kecimen, we first need to transform those labels into -1 and +1 for perceptron algorithm which requires class labels of -1, +1.

In [ ]:
# Perceptron requires labels in {-1, 1}.
# Class 0 (Besni) -> -1
# Class 1 (Kecimen) -> +1

# Check the unique classes before transformation
print("Original unique classes:", np.unique(y_np)) 

# Direct mapping from strings to Perceptron labels (-1, +1)
label_mapping = {'Besni': -1, 'Kecimen': 1} 

# Apply the mapping to create the final transformed labels
y_transformed = np.array([label_mapping[label] for label in y_np])

# Final verification
print("Transformed unique classes:", np.unique(y_transformed))

#### Training and Validation Split

1. Explain why it’s important to use the validation set to evaluate your model instead
of relying solely on the training data.
2. Discuss how evaluating the model on unseen
data helps in assessing its generalization capability.

When building a machine learning model, it is important to evaluate its performance not only on the data it was trained on but also on unseen data.  
To achieve this, we divide the dataset into two parts:

- Training set — used by the perceptron to learn patterns and adjust weights. Generally 80% of the data.
- Validation set — used to evaluate how well the trained model performs on new, unseen samples. Generally 20% of the data.

If we only evaluate the model on the training data, we might get a perfect result because the model could simply memorize patterns too specific to that data and the noise and it is called **overfitting**.
By testing on the validation set, we can measure how effectively the model can make predictions on data it has never seen before. which we call **generalization**. The reason behind this is perceptron will only get trained by the training set, it will never see validation set and the y values on it. Then when we test the trained model we will see if the perceptron can make good predictions on brand new data.

In [ ]:
# Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_np, y_transformed, test_size=0.2, random_state=23, shuffle=True
)

# Print the shapes of the resulting sets
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of Y_train:", y_train.shape) 
print("Shape of Y_test:", y_test.shape)   

Before applying feature scaling, we first will splitted the data. This ensures that the scaling process does not see information from the validation data, which would lead to data leakage.

Below we will define the preprocessing methods, then use the scaled data in the **next section** and see the accuracy results of all methods.

In [ ]:
def standardize(X_train, X_test):
    # X_scaled = (X - mean) / std_dev     
    # Calculate mean and standard deviation
    mean_train = np.mean(X_train, axis=0)
    std_train = np.std(X_train, axis=0)

    # avoid division by zero
    std_train_safe = np.where(std_train == 0, 1e-8, std_train)

    # apply to both sets 
    x_train_standardize = (X_train - mean_train) / std_train_safe
    x_test_standardize = (X_test - mean_train) / std_train_safe

    return x_train_standardize, x_test_standardize

In [ ]:
def minmax_scale(X_train, X_test):
    # X_scaled = (X - X_min) / (X_max - X_min)
    
    # calculate min and max only from the training data
    min_train = np.min(X_train, axis=0)
    max_train = np.max(X_train, axis=0)
    
    # Calculate the range (denominator)
    range_train = max_train - min_train
    
    # avoid division by zero
    range_train[range_train == 0] = 1e-8 
    
    # apply to both sets 
    X_train_minmax_scaled = (X_train - min_train) / range_train
    X_test_minmax_scaled = (X_test - min_train) / range_train
    
    return X_train_minmax_scaled, X_test_minmax_scaled

In [ ]:
def robust_scale(X_train, X_test):
    # X_scaled = (X - Median) / IQR

    # calculate median
    median_train = np.median(X_train, axis=0)
    
    # calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1_train = np.percentile(X_train, 25, axis=0)
    Q3_train = np.percentile(X_train, 75, axis=0)
    
    # calculate the Interquartile Range (IQR)
    IQR_train = Q3_train - Q1_train
    
    # avoid division by zero
    IQR_train[IQR_train == 0] = 1e-8
    
    # apply to both sets 
    X_train_robust_scaled = (X_train - median_train) / IQR_train
    X_test_robust_scaled = (X_test - median_train) / IQR_train
    
    return X_train_robust_scaled, X_test_robust_scaled

### 3. Implementing Perceptron for Binary Classification (20 points)

The Perceptron Learning Algorithm is applied to the Raisin dataset because the task is a fundamental binary classification problem (Kecimen vs. Besni). The Perceptron is the simplest model designed to find a linear separator (or hyperplane) between two classes. Using the Perceptron allows us to establish a baseline and determine if the classification boundary for the seven features can be defined by a simple, straight line.

When the dataset is not linearly separable, the Perceptron is not guaranteed to converge. Instead of finding a single, perfect decision boundary (which would result in 100\% training accuracy), the weights and the boundary will perpetually oscillate. The algorithm would continue updating its weights indefinitely (or until the maximum number of epochs is reached), never achieving zero classification errors. Its final performance would be limited to the highest accuracy achieved just before oscillation pushed the boundary into a worse configuration.

####  Implement Percepton Learning Algorithm

In [ ]:
class Perceptron:
    def __init__(self, learning_rate=0.01, n_epochs=50, random_state=20):
        # Initialize the model parameters (weights, bias) and hyperparameters
        self.lr = learning_rate
        self.n_epochs = n_epochs
        # Setting random_state for reproducibility in NumPy's random functions
        self.random_state = random_state 
        self.weights = None
        self.bias = None
        self.errors_ = []  # To store misclassification count per epoch

    def fit(self, X, y):
        rgen = np.random.RandomState(self.random_state)
        
        # Initialize weights to small random numbers, and bias to 0.0
        # X.shape[1] is the number of features
        self.weights = rgen.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.bias = 0.0
        
        # Iterate over epochs
        for _ in range(self.n_epochs):
            errors = 0
            # Iterate over each training sample
            for xi, target in zip(X, y):
              
                # Check for misclassification: 
                # The update only occurs if y_i and prediction have different signs
                prediction = self.predict(xi)
                
                if target != prediction:
                    # Target is the true label (y_i in {-1, 1})
                    # xi is the input vector

                    # Calculate prediction: 
                    update = self.lr * target # update is eta * y_i

                    # Update weights: w <- w + eta * y_i * x_i 
                    self.weights += update * xi
                    
                    # Update bias: b <- b + eta * y_i 
                    self.bias += update
                    
                    errors += 1
            
            self.errors_.append(errors)
            
            # Add an early stop if errors == 0 (convergence)
            if errors == 0:
                print("Model converged")
                break
                
        return self

    def weighted_sum(self, X):
        # Calculate the weighted sum z = w^T * x + b
        # The result must be a 1D array of z values (one for each row in X)
        return np.dot(X, self.weights) + self.bias

    def predict(self, X):
        # Determine the class label based on the sign of z (Step Function)
        return np.where(self.weighted_sum(X) >= 0.0, 1, -1)

In the perceptron model above, and perceptron model in general has several parameters that control how it learns from data.  

`Learnable parameters` are initialized once and then iteratively adjusted during the training process to minimize classification errors.

 **1. Weights (self.weights)**
- Represent the strength of the connection between each input feature and the output.
- Initially set to small random values so that not all features learn the same way.
- During training, they are updated whenever a misclassification occurs using the rule: w = w + eta * y_i * x_i
- The weights determine the orientation of the hyperplane that separates the two classes.

 **2. Bias (self.bias)**
- A constant term added to the weighted sum to shift the decision boundary.
- Updated similarly to the weights when a misclassification occurs: b = b + eta * y_i
- Allows the hyperplane to move away from the origin, improving flexibility in classification.

`Hyperparameters` control the behavior and optimization process of the algorithm but are not learned from the data itself.

 1. **Learning Rate (learning_rate)**
- Controls the step size for each update to the weights and bias.
- If eta is too large updates overshoot the optimal boundary and cause oscillation.
- If eta is too small training becomes very slow and might get stuck before convergence.
- 0.001–0.1 depending on the dataset and scaling.

 **2. Number of Epochs (n_epochs)**
- Specifies how many passes the algorithm makes over the entire training dataset.
- Each epoch processes all training samples once.
- A higher number gives the model more chances to learn, but if too high, it may lead to unnecessary computation or overfitting.

 **3. Random State (random_state)**
- Used to seed the random number generator (np.random.RandomState).
- Ensures that weight initialization is the same each time the code run so we get consistent results everytime we rerun. (reproducability)

By tuning these parameters, we can control the speed, stability, and accuracy of the Perceptron’s training process.


#### Accuracy implementation

In [ ]:
def calculate_accuracy(y_true, y_pred):
    """
    Calculates the accuracy score.
    y_true: true labels (NumPy array)
    y_pred: predicted labels (NumPy array)
    """
    # Compare true labels and predicted labels. 
    # np.sum counts the number of 'True' comparisons (correct predictions).
    correct_predictions = np.sum(y_true == y_pred)
    total_samples = len(y_true)
    
    accuracy = correct_predictions / total_samples
    return accuracy

#### Train the Perceptron

Training the perceptrons, predictions with the accuracy on training set and different preprocessing methods' effects are all implemented and explained below.

#### Prediction on Training Data

In [ ]:
# Apply standardization
X_train_standardize, X_test_standardize = standardize(X_train, X_test)

# Instantiate and train Perceptron
ppn_standardize = Perceptron() 
ppn_standardize.fit(X_train_standardize, y_train) 

# Predict and evaluate
y_train_pred_standardize = ppn_standardize.predict(X_train_standardize)
accuracy_standardize = calculate_accuracy(y_train, y_train_pred_standardize)

print(f"Standardized Training Accuracy: {accuracy_standardize:.4f}")

In [ ]:
y_test_pred_standardize = ppn_standardize.predict(X_test_standardize)
accuracy_minmax = calculate_accuracy(y_test, y_test_pred_standardize)

print(f"Standardized Test Accuracy: {accuracy_minmax:.4f}")

In [ ]:
# Apply Min-Max Scaling
X_train_minmax, X_test_minmax = minmax_scale(X_train, X_test)

# Instantiate and train Perceptron with Min-Max scaled data
ppn_minmax = Perceptron() 
ppn_minmax.fit(X_train_minmax, y_train) 

# Predict and evaluate
y_train_pred_minmax = ppn_minmax.predict(X_train_minmax)
accuracy_minmax = calculate_accuracy(y_train, y_train_pred_minmax)

print(f"Min-Max Training Accuracy: {accuracy_minmax:.4f}")

In [ ]:
y_test_pred_minmax = ppn_minmax.predict(X_test_minmax)
accuracy_minmax = calculate_accuracy(y_test, y_test_pred_minmax)

print(f"Min-Max Test Accuracy: {accuracy_minmax:.4f}")

In [ ]:
# Apply Robust Scaling
X_train_robust, X_test_robust = robust_scale(X_train, X_test)

# Instantiate and train Perceptron with Robust scaled data
# Use your desired hyperparameters
ppn_robust = Perceptron() 
ppn_robust.fit(X_train_robust, y_train) 

# Predict and evaluate
y_train_pred_robust = ppn_robust.predict(X_train_robust)
accuracy_robust = calculate_accuracy(y_train, y_train_pred_robust)

print(f"Robust Scaled Training Accuracy: {accuracy_robust:.4f}")

In [ ]:
y_test_pred_robust = ppn_robust.predict(X_test_robust)
accuracy_robust = calculate_accuracy(y_test, y_test_pred_robust)

print(f"Robust Scaled Test Accuracy: {accuracy_robust:.4f}")

In [ ]:
def train_perceptron_hyperparameters(X_train, y_train):
    # Use a range of values for learning rate and epochs
    etas = [0.1, 0.01, 0.001, 0.0001]
    epochs = [1, 10, 100, 500, 1000, 10000] 

    best_accuracy = 0.0
    best_params = {}
    best_ppn = None
    results_list = []
    print("--- Starting Hyperparameter Search ---")

    for lr in etas:
        for epoch in epochs:
            # 1. Instantiate the Perceptron
            # Use a fixed random_state for consistency
            ppn_candidate = Perceptron(learning_rate=lr, n_epochs=epoch)
            
            # 2. Train the model
            ppn_candidate.fit(X_train, y_train)
            
            # 3. Predict on training data
            y_pred_train_candidate = ppn_candidate.predict(X_train)
            
            
            # 4. Calculate Accuracy (using the already defined function)
            current_accuracy = calculate_accuracy(y_train, y_pred_train_candidate)
            
            results_list.append({
                'LR': lr, 
                'Epochs': epoch, 
                'Accuracy': current_accuracy
            })

            print(f"LR: {lr:.4f}, Epochs: {epoch:4d} -> Acc: {current_accuracy:.4f}")
            
            # 5. Track the best model
            if current_accuracy > best_accuracy:
                best_accuracy = current_accuracy
                best_params = {'learning_rate': lr, 'n_epochs': epoch}
                best_ppn = ppn_candidate
        print("\n")
    print("--- Search Complete ---")
    print(f"Best Training Accuracy: {best_accuracy:.4f} found with parameters: {best_params}")
    return best_ppn, results_list

In [ ]:
best_ppn, search_results = train_perceptron_hyperparameters(X_train_robust, y_train)

In [ ]:
# 1. Create DataFrame directly from the returned list
df = pd.DataFrame(search_results)

# 2. Pivot the DataFrame to create the grid for the heatmap
pivot_table = df.pivot(index='LR', columns='Epochs', values='Accuracy')

# 3. Identify the best accuracy for highlighting
best_acc = df['Accuracy'].max()

# 4. Generate the heatmap plot
plt.figure(figsize=(10, 6))
plt.imshow(pivot_table, cmap='viridis', aspect='auto')

# Labels and Ticks
plt.colorbar(label='Training Accuracy')
plt.xlabel('Number of Epochs')
plt.ylabel('Learning Rate')
plt.title('Perceptron Training Accuracy by Hyperparameters (Robust Scaled)')

# Set ticks and labels for Epochs (X-axis)
epochs_labels = pivot_table.columns.tolist()
plt.xticks(np.arange(len(epochs_labels)), epochs_labels, rotation=45, ha='right')

# Set ticks and labels for LR (Y-axis)
lr_labels = pivot_table.index.tolist()
lr_labels_str = [f'{lr:.4f}' for lr in lr_labels]
plt.yticks(np.arange(len(lr_labels)), lr_labels_str) 

# 5. Add text annotations for accuracy values
for i in range(len(lr_labels)):
    for j in range(len(epochs_labels)):
        acc = pivot_table.iloc[i, j]
        # Highlight the best accuracy
        color = 'red' if acc == best_acc else 'white'
        fontweight = 'bold' if acc == best_acc else 'normal'
        plt.text(j, i, f'{acc:.4f}', ha='center', va='center', color=color, fontsize=8, fontweight=fontweight)

plt.tight_layout()

plt.show()

##### Hyperparameter Effect on Training Process

**'Learning Rate'** controls the step size taken in the direction of error correction. It determines the speed and stability of the convergence process.

**'Number of Epochs'**,"Defines the maximum number of full passes over the training data. If the data is separable, this dictates how long it takes to achieve 100% accuracy. If non-separable, it acts as the necessary stopping condition before endless oscillation.

**Effect of Learning Rate Extremes**

1.  Very Large Learning Rate :
    The model takes very large steps, often overshooting the minimum error point in the weight space . This leads to instability and oscillation, preventing the model from ever settling on an optimal boundary.

    Our Results: The LR=0.1000 shows this instability; it peaked early at Acc=0.8375 (100 epochs) but then saw accuracy drop, reflecting  oscillation.
    
2.  Very Small Learning Rate :
    The model takes tiny steps, leading to a much slower training process. While stable, it may take an impractical amount of time or epochs to reach the best possible boundary.
    
    Our Results:  The best overall accuracy of 0.8597 was achieved with the smallest learning rate (LR=0.0001), but it required a high number of epochs (Epochs=1000). This configuration maximized accuracy by using many small adjustments to carefully navigate the weight space and stabilize at the most favorable separation boundary.

Looking at the hyperparameter search results, the training accuracies for all 24 combinations are incredibly close. They all bunch up between 0.80 and 0.86.This small range tells us the model is not finding a clean separation, meaning the Raisin data is almost certainly not linearly separable. If it were separable, we'd expect some combination to hit 100.

Because the model can't fully converge, the weight vector just ends up oscillating . Our 'best' tuning result isn't a true optimum; it's just the point where the oscillating decision boundary happened to be in the most favorable position when we stopped the training loop. This makes our hyperparameter tuning process less about optimizing and more about finding the best random stopping point before the oscillation makes things worse

In [ ]:
"""
 Here we assign our best model to the training and test data to use later 
 in the next section evaluation.
"""
y_train_pred = best_ppn.predict(X_train_robust)
y_test_pred = best_ppn.predict(X_test_robust)

##### 1. What does accuracy tell us about the performance of the model?

Accuracy measures the proportion of samples that were correctly classified, including both true positives and true negatives, out of all the samples.

Accuracy = (Number of Correct Predictions) / (Total Number of Predictions)

It gives a simple, overall indication of how often the model is right. For our model, an accuracy of 0.8444 on the validation set means that the model correctly classified about 84.44 percent of the 180 unseen samples.

##### 2. Why is accuracy sometimes not enough, especially in cases where the data is imbalanced? Explain a scenario where a high accuracy might be misleading.

Accuracy is often insufficient, especially when dealing with imbalanced data.

Accuracy can be a misleading performance metric when a dataset is imbalanced, meaning one class significantly outnumbers the other. A model can achieve high accuracy by simply learning to always predict the majority class, thereby failing to identify the critical minority class.

For a misleading scenario we can consider a medical test for a disease present in 1% of a population of 10,000. A model that blindly predicts "No Disease" for every patient would be correct for 9,900 healthy individuals, yielding a 99% accuracy. However, this model has a catastrophic 0% recall for the disease class, failing to identify any of the 100 sick patients. This demonstrates why metrics like precision, recall, and the F1-score are essential for a complete assessment in such contexts.

##### 3. In what types of problems is precision more important than recall?

Precision is more important than recall in scenarios where the cost of a false positive (FP) is unacceptably high. The goal is to ensure that when the model makes a positive prediction, we can trust it to be correct, even if it means missing some actual positive cases (lower recall).

An example might be a spam email detection. In this context, a false positive is a legitimate email incorrectly classified as spam. The consequence is that an important message like a a job offer is lost from the user's inbox. This is far more costly than a false negative, which is a spam email that slips into the inbox. Therefore, we prioritize high precision to guarantee that the emails sent to the spam folder are almost certainly spam.

##### 4. In what cases is recall more critical?

Recall is more critical than precision in scenarios where the cost of missing a positive instance (a false negative) is unacceptably high. The primary goal is to identify as many of the actual positive cases as possible, even if this means accepting a higher number of false alarms (lower precision).

Medical screening is a good example for this case. In a cancer screening test, a false negative means telling a sick patient they are healthy, which could lead to a lack of treatment. A false positive, while causing further testing, is a less critical error. Therefore, high recall is essential to ensure that nearly all patients with the disease are identified for the next stage of diagnosis.

##### 5. When is the F1 score a better measure than accuracy? What are the limitations of using F1 score alone?

The F1 score is the harmonic mean of precision and recall. It is generally a better measure than accuracy when the dataset is imbalanced, because it accounts for both precision (avoiding false alarms) and recall (catching all positive cases). The F1 score is only high when both precision and recall are high.

The main limitation of the F1 score is that it only focuses on the positive class (Kecimen in our case).

1.  Ignores true negatives: A model can perform poorly on the negative class (Besni) but still show a decent F1 score, because true negatives are not included in the F1 calculation.
2.  Contextual ambiguity: The F1 score is a summary. For example, a model with 0.80 precision and 0.80 recall has the same F1 score as a model with 0.90 precision and 0.72 recall. Without looking at precision and recall values separately, we can't see which type of error is more common.

### 4. Evaluating Perceptron for Binary Classification (25 points)

#### Prediction on Validation Data

In [ ]:
def calculate_metrics(y_true, y_pred):
    """
    Calculates Accuracy, Precision, Recall, and F1 Score, and returns Confusion Matrix components.
    The labels y_true and y_pred must be in the format {-1, +1}.
    """
    # Convert to 1D arrays if necessary
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()

    # Define the Positive class as +1 (Kecimen), and Negative class as -1 (Besni)
    
    # Calculate the Confusion Matrix Components
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == -1) & (y_pred == -1))
    FP = np.sum((y_true == -1) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == -1))
    
    total_samples = len(y_true)
    
    #  Calculate Metrics 
    accuracy = (TP + TN) / total_samples
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    support = TP + FN
    
    return accuracy, precision, recall, f1, support, TP, TN, FP, FN

In [ ]:
# --- 1. Training Metrics ---
accuracy_train, precision_train, recall_train, f1_train, support_train, TP_tr, TN_tr, FP_tr, FN_tr = calculate_metrics(y_train.flatten(), y_train_pred)

print("========== Training Metrics ==========")
print(f"Training Accuracy:    {accuracy_train:.4f}")
print(f"Training Precision:   {precision_train:.4f}")
print(f"Training Recall:      {recall_train:.4f}")
print(f"Training F1 Score:    {f1_train:.4f}")
print(f"Support (+1 Class):   {support_train}")
print("======================================\n")

# --- 2. Validation Metrics (Test Set) ---
accuracy_test, precision_test, recall_test, f1_test, support_test, TP_te, TN_te, FP_te, FN_te = calculate_metrics(y_test.flatten(), y_test_pred)

print("======== Validation Metrics ==========")
print(f"Validation Accuracy:  {accuracy_test:.4f}")
print(f"Validation Precision: {precision_test:.4f}")
print(f"Validation Recall:    {recall_test:.4f}")
print(f"Validation F1 Score:  {f1_test:.4f}")
print(f"Support (+1 Class):   {support_test}") 
print("======================================\n")

# --- 3. Confusion Matrix ---
print("====== Validation Confusion Matrix ======")
print("Predicted:   Kecimen (+1) | Besni (-1)")
print(f"Actual Kecimen (+1): {TP_te:11d} | {FN_te:7d}")
print(f"Actual Besni (-1):   {FP_te:11d} | {TN_te:7d}")
print("=======================================")

In [ ]:
def plot_confusion_matrix(TP, TN, FP, FN, title='Validation Confusion Matrix'):
    """
    Generates a heatmap plot of the Confusion Matrix using the standard [TN, FP], [FN, TP] layout.
    """
    # Create the 2x2 matrix: 
    # Row 0: Actual Negative (Besni)
    # Row 1: Actual Positive (Kecimen)
    cm = np.array([[TP, FN], 
                   [FP, TN]])
    
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    
    ax.set_title(title, fontsize=16)
    fig.colorbar(im)
    
    classes = ['Kecimen (+1)', 'Besni (-1)'] 
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes, fontsize=12) # Predicted labels (Columns)
    ax.set_yticklabels(classes, fontsize=12) # Actual labels (Rows)
    
    ax.set_ylabel('Actual Label', fontsize=14)
    ax.set_xlabel('Predicted Label', fontsize=14)

    # Add annotations (numbers) inside the boxes
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm[i, j] > thresh else "black"
            ax.text(j, i, format(cm[i, j], 'd'),
                    ha="center", va="center",
                    color=color, fontsize=18)
            
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(
    TP=TP_te, 
    TN=TN_te, 
    FP=FP_te, 
    FN=FN_te, 
    title='Perceptron Confusion \n Matrix on Validation Set'
)


#### Interpretation of Key Metrics

##### 1. Accuracy

* **Representation:** The overall proportion of correctly classified samples out of the total 180 validation samples.
* **Result Interpretation:** An accuracy of 0.8444 is strong, meaning the model correctly classifies approximately 84 out of every 100 unseen raisins.
* **Strength/Weakness:** This is a good general performance indicator, and its closeness to the Training Accuracy (0.8597) suggests the model generalized well and did not severely overfit the training data.

##### 2. Precision 

* **Representation:** Measures the quality of the positive predictions. It is the probability that a sample is actually Kecimen, given that the model predicted it was Kecimen.
* **Formula:** Precision = True Positives / (True Positives + False Positives)
* **Result Interpretation:** A precision of 0.8485 means that when the Perceptron flagged a raisin as Kecimen, it was wrong only about 15.15% of the time (False Positives: 15 errors out of 99 positive predictions).
* **Strength/Weakness:** This is a strong aspect. It indicates that the model is generally reliable at avoiding false alarms (labeling a Besni raisin as Kecimen).

##### 3. Recall 

* **Representation:** Measures the model's ability to find all positive samples. It is the probability that a sample is predicted as Kecimen, given that it is actually Kecimen.
* **Formula:** Recall = True Positives / (True Positives + False Negatives)
* **Result Interpretation:** A recall of 0.8660 means the model successfully identified 86.60% of all 97 actual Kecimen raisins.
* **Strength/Weakness:** This is the model's strongest metric. It indicates a high rate of detection for the positive class, showing the Perceptron is highly effective at minimizing False Negatives (FN), which is crucial for identifying the class of interest.

##### 4. F1-Score

* **Representation:** The harmonic mean of Precision and Recall. It provides a single metric that balances the trade-off between the two.
* **Result Interpretation:** An F1-Score of 0.8571 is very high.
* **Strength/Weakness:** The high F1-Score demonstrates that the model achieved a good balance, avoiding the pitfalls of optimizing only one metric (i.e., it is both reliable in its positive predictions and highly comprehensive in its detection).

##### 5. Support

* **Representation:** The count of actual occurrences of the positive class (Kecimen) in the validation set, which is 97.
* **Result Interpretation:** The value of $97$ tells us that $97$ of the $180$ validation raisins were Kecimen.

##### Confusion Matrix Analysis

The matrix confirms that the Perceptron, despite the non-linearly separable data, performed well at separating the two classes.


### 5. Visualization of the Hyperplane (40 + 5 points)


#### Step 1: Feature Selection Using Correlation Analysis (10 points)

1. Correlation Analysis: One effective way to reduce the number of features is by
analyzing the correlation between them. Highly correlated features may provide re-
dundant information, so you can select two less correlated features for visualization.
2. Compute the correlation matrix: Analyze the relationship between the all features to identify pairs of features with low
correlation.

In [ ]:
correlation_matrix = X_df.corr()

print("--- Feature Correlation Matrix ---")
print(correlation_matrix.round(2))

# Display the correlation matrix as a heatmap
plt.figure(figsize=(8, 6))
plt.imshow(correlation_matrix, cmap='coolwarm', interpolation='nearest')
plt.colorbar(label='Correlation Coefficient')
plt.title('Correlation Matrix of Raisin Features')

# Set ticks and labels
feature_names = X_df.columns.tolist()
plt.xticks(np.arange(len(feature_names)), feature_names, rotation=45, ha='right')
plt.yticks(np.arange(len(feature_names)), feature_names)

# Add annotations for key values (optional but helpful)
for i in range(len(feature_names)):
    for j in range(len(feature_names)):
        text = plt.text(j, i, correlation_matrix.iloc[i, j].round(2),
                        ha="center", va="center", color="black", fontsize=8)

plt.tight_layout()
plt.show()

3. Choose two features: Based on the correlation analysis, select two features that are
least correlated (e.g., variance and skewness) to visualize the Perceptron’s decision
boundary.

In [ ]:
feature1_name = 'Extent'
feature2_name = 'Eccentricity' 
features_2d_df = X_df[[feature1_name, feature2_name]]

# 1. Split the data 
X_2d_np = features_2d_df.to_numpy()
X_train_2d, X_test_2d, _, _ = train_test_split(
    X_2d_np, y_transformed, test_size=0.2, random_state=23, shuffle=True
)

# 2. Apply the best scaling method 
X_train_2d_scaled, X_test_2d_scaled = robust_scale(X_train_2d, X_test_2d)

#### Step 2: Train the Perceptron on Selected Features (10 points)

1. Retrain the Perceptron: After selecting two features, train the Perceptron using
only these two features. Use the same training and evaluation process as before,
but now the model will work in a two-dimensional feature space.

In [ ]:
# Use the best hyperparameters found previously 
best_lr = 0.0001
best_epochs = 100

# Instantiate a new Perceptron for the 2D feature space
ppn_2d = Perceptron(learning_rate=best_lr, n_epochs=best_epochs, random_state=20) 

# Train the model
ppn_2d.fit(X_train_2d_scaled, y_train)

# Predict on the training data
accuracy_train_2d = calculate_accuracy(y_train, ppn_2d.predict(X_train_2d_scaled))

print(f"Features: {feature1_name} and {feature2_name}")
print(f"2D Training Accuracy: {accuracy_train_2d:.4f}")

y_pred_test_2d = ppn_2d.predict(X_test_2d_scaled)

accuracy_test_2d, precision_test_2d, recall_test_2d, f1_test_2d= calculate_metrics(y_test, y_pred_test_2d)[:4]

print("\n--- 2D Validation Metrics ---")
print(f"Validation Accuracy: {accuracy_test_2d:.4f}")
print(f"Validation Precision: {precision_test_2d:.4f}")
print(f"Validation Recall: {recall_test_2d:.4f}")
print(f"Validation F1 Score: {f1_test_2d:.4f}")


2. Visualize the decision boundary (hyperplane): Once the Perceptron is trained, plot
the decision boundary (hyperplane) to see how the Perceptron separates the two
classes in 2D.

In [ ]:
def plot_decision_boundary(X, y, model, resolution=0.02):
    # Setup marker generator and color map
    markers = ('o', 's', 'x', '^', 'v')
    colors = ('blue', 'red', 'lightgreen', 'gray', 'cyan')
    cmap = ListedColormap(colors[:len(np.unique(y))])

    # Determine the range for the grid
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    
    # Create the mesh grid (x1 for feature 1, x2 for feature 2)
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, resolution),
                           np.arange(x2_min, x2_max, resolution))
    
    # Predict the class for every point on the grid (flatten the grid, predict, and reshape)
    Z = model.predict(np.array([xx1.ravel(), xx2.ravel()]).T)
    Z = Z.reshape(xx1.shape)
    
    # Plot the decision boundary contour
    plt.contourf(xx1, xx2, Z, alpha=0.3, cmap=cmap)
    plt.xlim(xx1.min(), xx1.max())
    plt.ylim(xx2.min(), xx2.max())

    # Plot the class samples
    for idx, cl in enumerate(np.unique(y)):
        plt.scatter(x=X[y == cl, 0], 
                    y=X[y == cl, 1],
                    alpha=0.8, 
                    c=colors[idx],
                    marker=markers[idx], 
                    label=f'Class {cl} ({feature1_name} vs {feature2_name})',
                    edgecolor='black')

    plt.title(f'Perceptron Decision Boundary (Acc: {accuracy_train_2d:.4f})', fontsize=14)
    plt.xlabel(feature1_name + ' (Scaled)', fontsize=12)
    plt.ylabel(feature2_name + ' (Scaled)', fontsize=12)
    plt.legend(loc='upper left')
    plt.show()


In [ ]:
# Plot the decision boundary using the selected features and trained model
plot_decision_boundary(X_train_2d_scaled, y_train, ppn_2d)

#### Step 3: Experiment with Different Features (10 points)

After visualizing the decision boundary for one pair of features, try selecting different
combinations of features (e.g., Area and MaxorAxisLength) and retrain the Perceptron. Compare
how the hyperplane changes with different features. This helps in understanding how the
Perceptron’s decision-making process adapts to different feature sets.

In [ ]:
feature_worst_1 = 'Area'
feature_worst_2 = 'ConvexArea'

features_worst_df = X_df[[feature_worst_1, feature_worst_2]]
X_worst_np = features_worst_df.to_numpy()

# Split data (must use same random state as the 7D split)
X_train_worst, X_test_worst, _, _ = train_test_split(
    X_worst_np, y_transformed, test_size=0.2, random_state=23, shuffle=True
)

# Apply Robust Scaling (Best Preprocessing)
X_train_worst_scaled, X_test_worst_scaled = robust_scale(X_train_worst, X_test_worst)

# Instantiate and train NEW Perceptron (using best hyperparameters)
ppn_worst = Perceptron(learning_rate=0.0001, n_epochs=1000, random_state=20) 
ppn_worst.fit(X_train_worst_scaled, y_train)

# Calculate Accuracy
accuracy_train_worst = calculate_accuracy(y_train, ppn_worst.predict(X_train_worst_scaled))
print(f"Features: {feature_worst_1} and {feature_worst_2}")
print(f"Worst Correlation Training Accuracy: {accuracy_train_worst:.4f}")

y_pred_test_2d = ppn_worst.predict(X_test_2d_scaled)

accuracy_test_2d, precision_test_2d, recall_test_2d, f1_test_2d= calculate_metrics(y_test, y_pred_test_2d)[:4]

print("\n--- 2D Validation Metrics ---")
print(f"Validation Accuracy: {accuracy_test_2d:.4f}")
print(f"Validation Precision: {precision_test_2d:.4f}")
print(f"Validation Recall: {recall_test_2d:.4f}")
print(f"Validation F1 Score: {f1_test_2d:.4f}")

feature1_name, feature2_name, accuracy_train_2d = feature_worst_1, feature_worst_2, accuracy_train_worst

plot_decision_boundary(X_train_worst_scaled, y_train, ppn_worst)

In [ ]:
feature_avg_1 = 'Extent'
feature_avg_2 = 'Perimeter'

features_avg_df = X_df[[feature_avg_1, feature_avg_2]]
X_avg_np = features_avg_df.to_numpy()

# Split data 
X_train_avg, X_test_avg, _, _ = train_test_split(
    X_avg_np, y_transformed, test_size=0.2, random_state=23, shuffle=True
)

# Apply Robust Scaling
X_train_avg_scaled, X_test_avg_scaled = robust_scale(X_train_avg, X_test_avg)

# Instantiate and train NEW Perceptron 
ppn_avg = Perceptron(learning_rate=0.0001, n_epochs=1000, random_state=20) 
ppn_avg.fit(X_train_avg_scaled, y_train)

# Calculate Accuracy
accuracy_train_avg = calculate_accuracy(y_train, ppn_avg.predict(X_train_avg_scaled))
print(f"Features: {feature_avg_1} and {feature_avg_2}")
print(f"Average Correlation Training Accuracy: {accuracy_train_avg:.4f}")

y_pred_test_2d = ppn_avg.predict(X_test_2d_scaled)

accuracy_test_2d, precision_test_2d, recall_test_2d, f1_test_2d= calculate_metrics(y_test, y_pred_test_2d)[:4]

print("\n--- 2D Validation Metrics ---")
print(f"Validation Accuracy: {accuracy_test_2d:.4f}")
print(f"Validation Precision: {precision_test_2d:.4f}")
print(f"Validation Recall: {recall_test_2d:.4f}")
print(f"Validation F1 Score: {f1_test_2d:.4f}")

feature1_name, feature2_name, accuracy_train_2d = feature_avg_1, feature_avg_2, accuracy_train_avg

plot_decision_boundary(X_train_avg_scaled, y_train, ppn_avg)


##### 1. How does the decision boundary change when you use different pairs of features?

The Perceptron model is a linear classifier, so its decision boundary is always a straight line (a hyperplane in 2D). However, the line's slope, orientation, and position in the feature space change completely depending on which features are used:

* Feature Influence: The weights learned by the Perceptron directly correspond to the two selected features, so the line rotates to maximize separation in that specific 2D plane.
* Separability Impact: For feature pairs where the classes are heavily mixed (like the Least Correlated Pair, Acc ≈ 0.61), the boundary is forced to cut through the overlapped region, resulting in a poor fit. For pairs offering a good balance of predictive power, the line can be positioned more cleanly between the clusters to minimize errors.

##### 2. Can you find a pair of features that leads to better separation between the two classes? Also explain why this feature pair causes the better separation.

| Pair Tested | Observed Training Accuracy | Observed Separation in Plot |
| :--- | :--- | :--- |
| **Least Correlated** (Extent & Eccentricity) | Acc = 0.6181 | Heavy overlap, classes widely scattered. |
| **Highest Correlation** (Area & ConvexArea) | Acc = 0.8306 | Tight cluster, clear separation along one axis. |
| **Second Least Correlated** (Extent & Perimeter) | Acc = 0.8319 | Distinct, separable clusters. |

The pair with the **Second Lowest Correlation** (Extent and Perimeter) led to the absolute best separation (Acc = 0.8319).

**Why this pair provides better separation:**

The Second Lowest Correlated Pair achieves the best performance because it strikes an optimal balance between feature redundancy and predictive power:

* Sufficient Independence: The features are independent enough from each other that, when combined, they offer distinct information for the classification boundary.
* High Predictive Relevance: Both features are strong individual predictors of the class label. This allows the Perceptron to use both dimensions to organize the data into separate clusters, which is very amenable to linear separation.

The Highest Correlated Pair (Acc ≈ 0.8306), while almost as good, succeeded mainly through the high relevance of a single dimension (size), whereas the best pair was more effective because it used two different strong predictive characteristics together.

##### 3. If the classes cannot be linearly separated in a given pair of features, can we still talk about a linear relationship between them? Explain your reasoning.

Yes. It's important to distinguish between two types of linear relationships:

1.  **Linear Relationship between Features (Correlation):** This is the correlation coefficient between the features. This is independent of the class label. A correlation of 0.0 means those features have no linear relationship with each other.
2.  **Linear Separability (Feature-to-Class Relationship):** This refers to whether a single straight line can perfectly divide the feature space based on the class label.

When the classes cannot be linearly separated (such as in the case with Acc=0.61), it just means that distinguishing Besni from Kecimen in that 2D feature space requires a non-linear function. The Perceptron's failure here suggests that a more complex classifier is needed. The existence of a linear relationship between the features is separate from whether a linear boundary can separate the classes.

#### Bonus Step 4: Incrementally Add Data (+ 5 points)

In [ ]:
# --- Configuration ---
# Use the best 2D scaled data and best hyperparameters
X_inc = X_train_avg_scaled
y_inc = y_train

best_lr = 0.0001
best_epochs = 1000 

# Define the data increments 
increment_percentages = np.arange(0.1, 1.01, 0.1) 
increment_percentages = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]

num_samples_total = len(X_inc)

In [ ]:
# Create a figure with multiple subplots to show the progression
n_plots = len(increment_percentages)
fig, axes = plt.subplots(2, n_plots // 2, figsize=(16, 8)) 
axes = axes.flatten()

# Re-use the plot_decision_boundary logic, but modify it to fit the subplot
def plot_incremental_boundary(X_subset, y_subset, model, ax, percentage):
    # Setup marker generator and color map
    markers = ('o', 's', 'x', '^', 'v')
    colors = ('blue', 'red')
    cmap = ListedColormap(colors)

    # Calculate the range for the grid based on the full dataset range
    x1_min, x1_max = X_inc[:, 0].min() - 0.5, X_inc[:, 0].max() + 0.5
    x2_min, x2_max = X_inc[:, 1].min() - 0.5, X_inc[:, 1].max() + 0.5
    resolution = 0.05
    
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, resolution),
                           np.arange(x2_min, x2_max, resolution))
    
    # Predict the class for every point on the grid using the current model
    Z = model.predict(np.array([xx1.ravel(), xx2.ravel()]).T)
    Z = Z.reshape(xx1.shape)
    
    # Plot the decision boundary contour
    ax.contourf(xx1, xx2, Z, alpha=0.3, cmap=cmap)
    
    # Plot the class samples (the points used for THIS step of training)
    for idx, cl in enumerate(np.unique(y_subset)):
        ax.scatter(x=X_subset[y_subset == cl, 0], 
                   y=X_subset[y_subset == cl, 1],
                   alpha=0.8, 
                   c=colors[idx],
                   marker=markers[idx], 
                   edgecolor='black',
                   s=20) # Small size for cleaner view
        
    ax.set_title(f'{int(percentage * 100)}% Data Used ({len(X_subset)} Samples)', fontsize=10)
    ax.set_xlim(x1_min, x1_max)
    ax.set_ylim(x2_min, x2_max)

# incremental learning loop
for i, percent in enumerate(increment_percentages):
    # Determine the number of samples for this step
    num_samples = int(num_samples_total * percent)
    
    # Select the subset of data
    X_subset = X_inc[:num_samples]
    y_subset = y_inc[:num_samples]
    
    # Create a new Perceptron instance for each step
    ppn_inc = Perceptron(learning_rate=best_lr, n_epochs=best_epochs, random_state=20)
    
    # Train the model on the small subset
    ppn_inc.fit(X_subset, y_subset)
    
    # Plot the result
    plot_incremental_boundary(X_subset, y_subset, ppn_inc, axes[i], percent)

# Display the final plot set
fig.suptitle('Perceptron Decision Boundary Evolution with Incremental Data', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

##### How does the decision boundary evolve as more data is added?

The decision boundary, or hyperplane, continuously evolves in an error-driven manner. In the early stages (low data percentages), the boundary is highly volatile and unstable. As the data progresses toward 100\%, the boundary gradually shifts to represent the best compromise across the entire data set, minimizing the total classification errors.

##### Does the Perceptron find a stable boundary after a certain number of data points?

The Perceptron never finds a truly fixed, stable boundary because the data is not linearly separable. However, the boundary becomes relatively stable once a critical mass of points (representing all major clusters) has been introduced. Beyond 80\% of the data, the shifts are typically minor and incremental, showing the model has learned the general linear separator and is only making small adjustments based on new boundary points.

##### How sensitive is the model to the order in which data points are introduced?

The model is highly sensitive to the order of data presentation. We observed clear evidence of this at the 80\% step, where the boundary made a radical, temporary shift that pushed the line far from the main clusters. This occurred because a few critical, misclassified points were introduced in that specific batch, forcing the model to make a massive, immediate correction. This highlights the Perceptron's dependence on the sequence of updates.

### Step 6: Fisher’s Linear Discriminant Projection (10 points)

In [ ]:
def fisher_ld_projection(X, y):
    # 1. Calculate the mean vectors μ0 and μ1 for each class
    # Class -1 (Besni)
    X_c0 = X[y == -1]
    mu_0 = np.mean(X_c0, axis=0)
    
    # Class +1 (Kecimen)
    X_c1 = X[y == 1]
    mu_1 = np.mean(X_c1, axis=0)
    
    # 2. Compute the within-class scatter matrix S_within
    # Calculate S_0
    S_0 = np.cov(X_c0, rowvar=False) * (len(X_c0) - 1)
    
    # Calculate S_1
    S_1 = np.cov(X_c1, rowvar=False) * (len(X_c1) - 1)
    
    S_within = S_0 + S_1
    
    # 3. Compute the between-class scatter term (equivalent to maximizing separation)
    mu_diff = mu_1 - mu_0
    
    # 4. Find the projection direction using w = S_within^-1 (μ1 - μ0) 
    S_within_inv = np.linalg.inv(S_within)
    w = np.dot(S_within_inv, mu_diff)
    
    # 5. Normalize the direction vector w to get a unit vector 
    w_unit = w / np.linalg.norm(w)
    
    return w_unit

def project_data(X, direction):
    # The dot product projects each data point (x) onto the direction vector (w)
    return np.dot(X, direction)

feature1 = 'Extent'
feature2 = 'Perimeter'
X_2d_unscaled = X_df[[feature1, feature2]].to_numpy() 

# need the labels for the entire set, not just the training set
y_full = y_transformed 

# split the data into classes for plotting purposes
X_c0 = X_2d_unscaled[y_full == -1]
X_c1 = X_2d_unscaled[y_full == 1]

In [ ]:
# Compute the optimal projection direction (w)
direction = fisher_ld_projection(X_2d_unscaled, y_full)

# Project the full dataset onto the Fisher LD direction
projected_data_full = project_data(X_2d_unscaled, direction)

# Separate projected data by class for the histogram
projected_data_c0 = projected_data_full[y_full == -1]
projected_data_c1 = projected_data_full[y_full == 1]

print(f"Fisher LD Direction Vector (w): {direction.round(3)}")

#### Visualizations
#### 2D Feature Space (Optional but Recommended)


In [ ]:
scale = 4

plt.figure(figsize=(8, 6))

# Plot the 2D data points
plt.scatter(X_c0[:, 0], X_c0[:, 1], color='blue', label='Besni (-1)', alpha=0.6)
plt.scatter(X_c1[:, 0], X_c1[:, 1], color='red', label='Kecimen (+1)', alpha=0.6)

# Calculate the mean of the entire dataset to position the vector at the center
mean_center = np.mean(X_2d_unscaled, axis=0)

# Visualize the projection direction vector (w)
plt.quiver(mean_center[0], mean_center[1], direction[0], direction[1], 
           scale=scale, color='black', width=0.005, 
           label='Fisher LD Direction (w)')

plt.title(f'Fisher LD Projection Direction in 2D Feature Space ({feature1} vs {feature2})', fontsize=14)
plt.xlabel(feature1, fontsize=12)
plt.ylabel(feature2, fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

#### 1D Projected Space (Required)

In [ ]:
plt.figure(figsize=(10, 6))

# Plot histograms for both classes
plt.hist(projected_data_c0, bins=30, alpha=0.6, label='Besni (-1)', color='blue')
plt.hist(projected_data_c1, bins=30, alpha=0.6, label='Kecimen (+1)', color='red')

plt.title('Distribution of Classes in 1D Fisher LD Projected Space', fontsize=14)
plt.xlabel('Projected Value (x ⋅ w)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.5)
plt.show()

##### 1. How well does Fisher’s LD separate the two classes in the 1D projected space?

Fisher's LD found the best possible straight line to look at the 2D raisin data. It succeeded in two ways:

It pulled the two class centers (Besni and Kecimen) as far apart as possible.It squished each class's data into a compact shape (low variance).

But, the main finding is: Even on this perfect line, the classes still overlap a lot in the middle, between -3.8 and -2.5. This overlap is the key problem. It means that there is no single point where we can draw a line and perfectly split Besni from Kecimen.

##### 2. Compare the separation of the two classes in the original 2D space (used in the Perceptron) versus the 1D Fisher’s LD projection. What do you observe about the class distributions in these spaces?

In the original 2D space, the two classes form large, overlapping clusters that the Perceptron struggled to separate cleanly, resulting in limited accuracy. After projecting onto 1D with Fisher's LD, the data becomes two tighter, more condensed distributions—but significant overlap remains between the classes. This demonstrates that, even with optimal linear separation in both spaces, the classes are still mixed in the middle. FLD provides the clearest possible view of this overlap, confirming that perfect separation isn’t possible for this data using linear methods.

##### 3. What insights can you gain from using Fisher’s LD for dimensionality reduction and class separation?

* We see that overlap is real. The Perceptron wasn't bad; the data just cannot be perfectly separated by a straight line, no matter how many dimensions we use (in a linear model). The large overlap in the histogram is the reason why our perceptron's accuracy is stuck at around 83% that overlap represents the minimum amount of error a linear model will always make. 
* We need a curve. To fully separate the classes and reach higher accuracy, we would need a non-linear classifier (a model that can draw a curve, not just a straight line) to wrap around the mixed points in the center.
* We also learned that we can simplify the data from two features down to one single score (the projected value) without losing any meaningful classification power.

### Conclusion

This project helped me lots to understand perceptron learning algorithm and its behavior. Also preprocessing methods for scaling, the metrics used for ml tasks, ways of separating data in different dimension and Fisher algorithm.

#### Some Key Findings

1.  **The Perceptron's Performance Limit:** We found that the 7-dimensional Raisin dataset is not linearly separable. The Perceptron's accuracy peaked around 86% on the training set. This lack of 100\% convergence, coupled with the erratic performance during hyperparameter tuning, showed the Perceptron constantly **oscillating** and unable to find a perfect boundary.

2.  **Best Configuration:** The final model, using **Robust Scaling** and the optimized hyperparameters (eta=0.0001, Epochs=1000), generalized well, achieving a solid 84.44 accuracy on the unseen validation set. The high F1-Score indicated the model was equally good at detecting the Kecimen class and avoiding false alarms.

3.  **Feature Impact:** Our 2D visualization experiments proved that selecting features isn't just about avoiding high correlation. The best separation (accuracy 83.19%) came from the pair (Extent & Perimeter) that offered the best balance of **independent information** and **predictive power**, allowing the linear boundary to function optimally.

4.  **The Overlap Problem (FLD Confirmation):** The Fisher's LD projection was the most insightful step. It found the absolute best single dimension to separate the classes. However, the resulting 1D histogram showed overlap between the two classes. This overlap is the proof of the constant error we get, confirming that our accuracy is capped by the data's fundamental non-linearity.


The Perceptron can successfully converge to an optimal boundary when the data is linearly separable, but in our case, it didn’t. It showed that the data couldn’t be perfectly separated by a straight line. Through this project, I learned how the Perceptron works, how convergence depends on the data, and how to use metrics and visualization methods like Fisher’s approach to better understand model behavior.